# 04 - K-Fold Cross Validation

Evaluate the KNN model robustly using Stratified K-Fold Cross Validation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.preprocess import handle_outliers, encode_categorical
from src.evaluate import calculate_metrics


### 1. Load and Preprocess Full Dataset


In [ ]:
# We need the full un-split data for K-Fold
df = pd.read_csv('../data/heart.csv')
df = df.drop_duplicates()
df = handle_outliers(df)
categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
df_encoded = encode_categorical(df, columns=categorical_cols)

X = df_encoded.drop(columns=['HeartDisease'])
y = df_encoded['HeartDisease']

numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']


### 2 & 3. Stratified K-Fold Loop


In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Best K from previous notebook (e.g., 17 or whatever was found)
best_k = 17 # Setting a default, but it would ideally be loaded or matched
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')

fold_metrics = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1 Score': [], 'Specificity': []}

for train_idx, test_idx in skf.split(X, y):
    X_train_fold, X_test_fold = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
    
    # Scale (fit ONLY on train fold)
    scaler = StandardScaler()
    X_train_fold[numeric_cols] = scaler.fit_transform(X_train_fold[numeric_cols])
    X_test_fold[numeric_cols] = scaler.transform(X_test_fold[numeric_cols])
    
    # Train and predict
    knn.fit(X_train_fold, y_train_fold)
    pred = knn.predict(X_test_fold)
    
    # Record metrics
    m = calculate_metrics(y_test_fold, pred)
    for k in fold_metrics.keys():
        fold_metrics[k].append(m[k])


### 4. Results after 10 folds


In [ ]:
print("--- 10-Fold CV Mean & Std ---")
for metric, values in fold_metrics.items():
    print(f"{metric}: Mean = {np.mean(values):.4f}, Std = {np.std(values):.4f}")

plt.figure(figsize=(6,4))
sns.boxplot(y=fold_metrics['Recall'])
plt.title('Recall Scores across 10 Folds')
plt.ylabel('Recall')
plt.show()


### 5. Quick validation with cross_val_score


In [ ]:
# Note: This is purely a quick check, but it doesn't handle scaling per fold natively unless placed in a Pipeline
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, metric='euclidean'))
])

# cross_val_score will handle the pipeline scaling correctly per fold
cv_recall = cross_val_score(pipeline, X, y, cv=skf, scoring='recall')
print(f"cross_val_score Mean Recall: {cv_recall.mean():.4f}")


### Key Insight

**Compare single split recall (notebook 3) vs K-Fold mean recall (notebook 4)**
- The single split might give a slightly higher or lower score depending on the random state (luck of the draw).
- **K-Fold is much more trustworthy** because it averages the performance across 10 different, non-overlapping test sets, ensuring that the model generalizes well across the entire dataset, not just one specific split.
